# ASAP8 longitudinal phase-shift analysis

This notebook isolates the "phase" phenomenon from the broader change/omission response question. The target property is a **reproducible event-locked temporal preference that changes across sessions in the same registered VIP neuron**.

The analysis is deliberately close to the data:
- no temporal smoothing or filtering of dF/F traces;
- all displayed voltage traces are raw extracted trial dF/F after event alignment only;
- dF/F phase is estimated from each trial's first harmonic over the 0.75-s stimulus cycle, after subtracting that trial's pre-event baseline;
- reproducible phasic structure is tested by randomizing trial phases while preserving each trial's harmonic amplitude;
- longitudinal phase differences are tested by permuting session labels across trial coefficients for the same registered cell;
- peak latency of the unsmoothed trial-mean dF/F and spike preferred phase are retained as independent, interpretable checks.

For image responses, an **identity-balanced phase** is also computed by giving each sufficiently sampled image identity equal weight. This helps distinguish a genuine timing shift from a change in which image identities dominate the trial pool.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, re, warnings
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import DEPTH_GROUP_ORDER, build_voltage_session_table, build_voltage_roi_table
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.voltage.responses import build_single_trial_index
from vip_slap2_analysis.voltage.spikes import DETECTOR_VERSION, extract_session_spikes

assert DETECTOR_VERSION == "template_v1"
sns.set_style("white")
plt.rcParams.update({"legend.fontsize":10,"axes.labelsize":13,"axes.titlesize":14,"xtick.labelsize":11,"ytick.labelsize":11})
display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:
# ------------------------------ Configuration ------------------------------
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MICE = [852835, 863774]
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
SESSION_ORDER = ["A0","A1","A2","B0","B1","B2"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0

STIMULUS_PERIOD_S = 0.75
PHASE_WINDOW_S = (0.0, STIMULUS_PERIOD_S)
BASELINE_WINDOW_S = (-0.25, 0.0)
MIN_PHASE_TRIALS = 8
MIN_SPIKES_FOR_PHASE = 10
MIN_IMAGE_TRIALS_PER_IDENTITY = 5
MIN_IDENTITIES_FOR_BALANCE = 3
N_PERM_LOCKING = 1000
N_BOOT_PHASE = 2000
N_PERM_SHIFT = 2000
FDR_ALPHA = 0.05
RNG_SEED = 23

SPIKE_KWARGS = dict(height_sigma=3.5, template_sigma=4.5, prominence_sigma=0.5)
RECOMPUTE_SPIKES = False
CACHE_DIR = Path.cwd() / "asap8_cell_response_cache"
SAVE_PATH = Path.cwd() / "asap8_phase_figures"
CACHE_DIR.mkdir(parents=True,exist_ok=True); SAVE_PATH.mkdir(parents=True,exist_ok=True)
SPIKE_CACHE = CACHE_DIR / "asap8_change_omission_spikes.pkl"

DEPTH_COLORS = {"<100 µm":"#EBA287","100–150 µm":"#d1e2b0",">150 µm":"#7bbcd5"}
EVENT_COLORS = {"image":"#606060","change":"#d95f02","omission":"#1b9e77"}

def depth_group_from_um(depth):
    depth=float(depth); return "<100 µm" if depth < 100 else ("100–150 µm" if depth <= 150 else ">150 µm")


## Dataset and spikes


In [ ]:
registry=VIPSessionRegistry.from_basepath(BASE_PATH)
sessions=build_voltage_session_table(registry,subject_ids=TARGET_MICE,paradigms=PARADIGMS,exclude_session_types=EXCLUDE_SESSION_TYPES,trace_variant=TRACE_VARIANT,expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC)
sessions=sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].copy(); sessions["session_id"]=sessions["session_id"].astype(str)
rois=build_voltage_roi_table(sessions,registration_filename=REGISTRATION_FILENAME,exclude_invalid_rois=EXCLUDE_INVALID_ROIS); rois["session_id"]=rois["session_id"].astype(str); rois["depth_um"]=pd.to_numeric(rois["depth_um"],errors="coerce"); rois["depth_group"]=rois["depth_um"].map(depth_group_from_um)
events=build_change_detection_events(sessions); events["session_id"]=events["session_id"].astype(str)
trial_index=build_single_trial_index(sessions,events); trial_index["session_id"]=trial_index["session_id"].astype(str)

raw=registry.sessions(subject_ids=TARGET_MICE,paradigms=PARADIGMS,exclude_session_types=EXCLUDE_SESSION_TYPES).copy(); raw["session_id"]=raw["session_id"].astype(str); trace_rows=[]
for _,row in raw.iterrows():
    if str(row["session_id"]) not in set(sessions["session_id"]): continue
    asset=registry.resolve_assets(row); p=Path(asset.derived_dir)/"voltage"/f"voltage_session_traces_{TRACE_VARIANT}.h5"; trace_rows.append(dict(session_id=str(asset.session_id),trace_h5=p))
sessions=sessions.merge(pd.DataFrame(trace_rows).drop_duplicates("session_id"),on="session_id",how="left"); sessions["spike_trace_available"]=sessions["trace_h5_y"].map(lambda p:isinstance(p,Path) and p.exists())

print(f"{len(sessions)} sessions · {int(rois['included'].sum())} included ROI observations")


In [ ]:
def finish_axis(ax):
    sns.despine(ax=ax); ax.tick_params(axis="both",labelsize=10)
    for spine in ax.spines.values(): spine.set_linewidth(1.5)

def decode_strings(values): return np.asarray([x.decode() if isinstance(x,(bytes,np.bytes_)) else str(x) for x in np.asarray(values).reshape(-1)])
def h5_roi_axis(group,dmd,source_roi):
    ids=decode_strings(group["roi_ids"][:]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"; hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(label)
    return int(hits[0])
def reconcile_timebase(t,n):
    t=np.asarray(t,float).reshape(-1)
    if len(t)==n:return t
    dt=float(np.nanmedian(np.diff(t))); zero=min(int(np.nanargmin(np.abs(t))),n-1); return (np.arange(n)-zero)*dt
def window_slice(t,window):
    i0=int(np.searchsorted(t,window[0],side="left")); i1=int(np.searchsorted(t,window[1],side="left")); return slice(i0,i1)
def align_trials_to_true_onset(traces,t,offsets):
    aligned=np.full_like(np.asarray(traces,float),np.nan,dtype=float)
    for i,(y,o) in enumerate(zip(traces,np.asarray(offsets,float))):
        if np.isfinite(o): aligned[i]=np.interp(t+o,t,y,left=np.nan,right=np.nan)
    return aligned

def indexed_event_table(session_id,dmd,event_type):
    sid=str(session_id); idx=trial_index[(trial_index["session_id"].astype(str)==sid)&(trial_index["dmd"].astype(int)==int(dmd))&(trial_index["event_type"].astype(str)==event_type)].copy()
    if "matched" in idx: idx=idx[idx["matched"].astype(bool)]
    idx=idx.rename(columns={"onset_sec":"stored_onset_sec","image_name":"stored_image_name"}); ev=events[events["session_id"].astype(str)==sid].copy()
    needed=["session_id","event_id","onset_sec","image_label","is_change","is_omission","sequence_position_expected"]
    for c in needed:
        if c not in ev:ev[c]=np.nan
    ev=ev[needed].rename(columns={"onset_sec":"cycle_onset_sec","image_label":"event_image_label"})
    return idx.merge(ev,on=["session_id","event_id"],how="left",validate="many_to_one").sort_values("trial_index").reset_index(drop=True)

def bh_fdr(p):
    p=np.asarray(p,float); out=np.full(len(p),np.nan); good=np.isfinite(p); vals=p[good]; m=len(vals)
    if not m:return out
    order=np.argsort(vals); q=vals[order]*m/np.arange(1,m+1); q=np.minimum.accumulate(q[::-1])[::-1]; q=np.clip(q,0,1); tmp=np.empty(m); tmp[order]=q; out[good]=tmp; return out

def circular_diff(a,b,period=STIMULUS_PERIOD_S): return ((np.asarray(a)-np.asarray(b)+period/2)%period)-period/2

def coeff_to_latency(z,period=STIMULUS_PERIOD_S):
    if not np.isfinite(np.real(z)) or not np.isfinite(np.imag(z)) or abs(z)==0:return np.nan
    return float((-np.angle(z)/(2*np.pi)*period)%period)

def trial_harmonic_coefficients(t,traces):
    t=np.asarray(t,float); traces=np.asarray(traces,float); bsl=window_slice(t,BASELINE_WINDOW_S); ph=window_slice(t,PHASE_WINDOW_S); tp=t[ph]; omega=2*np.pi/STIMULUS_PERIOD_S
    baseline=np.nanmean(traces[:,bsl],axis=1); seg=traces[:,ph]-baseline[:,None]; basis=np.exp(-1j*omega*tp)[None,:]; good=np.isfinite(seg); denom=np.sum(good,axis=1); coeff=np.full(len(seg),np.nan+1j*np.nan,dtype=complex)
    ok=denom>2; coeff[ok]=2*np.nansum(seg[ok]*basis,axis=1)/denom[ok]
    return coeff

def phase_locking_p(coeff,n_perm=N_PERM_LOCKING,seed=RNG_SEED):
    coeff=np.asarray(coeff,complex); coeff=coeff[np.isfinite(coeff.real)&np.isfinite(coeff.imag)]
    if len(coeff)<2:return np.nan
    obs=abs(np.mean(coeff)); rng=np.random.default_rng(seed); null=np.empty(int(n_perm))
    amp=np.abs(coeff)
    for i in range(int(n_perm)):
        null[i]=abs(np.mean(amp*np.exp(1j*rng.uniform(0,2*np.pi,len(coeff)))))
    return float((1+np.sum(null>=obs))/(len(null)+1))

def bootstrap_phase_ci(coeff,n_boot=N_BOOT_PHASE,seed=RNG_SEED):
    coeff=np.asarray(coeff,complex); coeff=coeff[np.isfinite(coeff.real)&np.isfinite(coeff.imag)]
    if len(coeff)<2:return (np.nan,np.nan,np.nan)
    center=coeff_to_latency(np.mean(coeff)); rng=np.random.default_rng(seed); phases=np.empty(int(n_boot))
    for i in range(int(n_boot)): phases[i]=coeff_to_latency(np.mean(rng.choice(coeff,size=len(coeff),replace=True)))
    d=circular_diff(phases,center); lo,hi=np.quantile(d,[.025,.975]); return center,float(lo),float(hi)

def rayleigh_p(phases):
    phases=np.asarray(phases,float); phases=phases[np.isfinite(phases)]; n=len(phases)
    if n<2:return np.nan
    R=abs(np.mean(np.exp(1j*phases))); z=n*R**2
    p=np.exp(-z)*(1+(2*z-z**2)/(4*n)-(24*z-132*z**2+76*z**3-9*z**4)/(288*n**2)) if n<50 else np.exp(-z)
    return float(np.clip(p,0,1))


In [ ]:
# Load/reuse spikes from the response-screen notebook when possible.
if SPIKE_CACHE.exists() and not RECOMPUTE_SPIKES:
    spikes=pd.read_pickle(SPIKE_CACHE); print(f"Loaded {len(spikes):,} cached spikes")
else:
    tables=[]
    for session in sessions.itertuples(index=False):
        if not bool(session.spike_trace_available): continue
        rr=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(str(session.session_id))]; lookup={int(d):g["roi"].astype(int).tolist() for d,g in rr.groupby("dmd")}
        table=extract_session_spikes(session.trace_h5_y,rois=lookup,**SPIKE_KWARGS); table.insert(0,"session_id",str(session.session_id)); tables.append(table)
    spikes=pd.concat(tables,ignore_index=True) if tables else pd.DataFrame()
    if len(spikes): spikes=spikes.merge(rois[["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]],on=["session_id","dmd","roi"],how="left")
    spikes.to_pickle(SPIKE_CACHE)


## Build unsmoothed trial phase coefficients


In [ ]:
# Cache aligned raw trials so the same data can be plotted and used in longitudinal tests.
TRIAL_CACHE={}

def load_cell_trials(session,row,event_type):
    key=(str(row.session_id),int(row.dmd),int(row.roi),str(event_type))
    if key in TRIAL_CACHE:return TRIAL_CACHE[key]
    with h5py.File(session.single_trial_h5,"r") as h5:
        g=h5[f"DMD{int(row.dmd)}"]; axis=h5_roi_axis(g,int(row.dmd),int(row.roi))
        if event_type=="image":
            idx=indexed_event_table(str(row.session_id),int(row.dmd),"image"); idx=idx[(~idx["is_change"].fillna(False).astype(bool))&(~idx["is_omission"].fillna(False).astype(bool))].copy(); stored_t=np.asarray(h5["timebase_sec/image"][:],float); t=stored_t.copy(); parts=[]; meta=[]
            for path,q in idx.groupby("dataset_path",sort=False):
                q=q.sort_values("trial_index"); ds=h5[str(path)]; t=reconcile_timebase(stored_t,int(ds.shape[-1])); parts.append(np.asarray(ds[q["trial_index"].astype(int).to_numpy(),axis,:],float)); meta.append(q)
            traces=np.concatenate(parts,axis=0) if parts else np.empty((0,len(stored_t))); idx=pd.concat(meta,ignore_index=True) if meta else idx.iloc[0:0]
            onsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy(float); labels=idx["event_image_label"].astype(str).to_numpy()
        else:
            traces=np.asarray(g[event_type]["traces"][:,axis,:],float); t=reconcile_timebase(h5[f"timebase_sec/{event_type}"][:],traces.shape[-1]); idx=indexed_event_table(str(row.session_id),int(row.dmd),event_type)
            if len(idx)!=len(traces):raise ValueError(f"{event_type} mismatch for {row.session_id} DMD{row.dmd}")
            offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy(); traces=align_trials_to_true_onset(traces,t,offsets); onsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy(float); labels=idx["event_image_label"].astype(str).to_numpy()
    TRIAL_CACHE[key]=(t,traces,onsets,labels); return TRIAL_CACHE[key]

phase_rows=[]
for session in sessions.itertuples(index=False):
    rr=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(str(session.session_id))]
    for r in rr.itertuples(index=False):
        for event_type in ["image","change","omission"]:
            t,traces,onsets,labels=load_cell_trials(session,r,event_type)
            if len(traces)<MIN_PHASE_TRIALS: continue
            coeff=trial_harmonic_coefficients(t,traces); valid=np.isfinite(coeff.real)&np.isfinite(coeff.imag); coeff=coeff[valid]; labels_v=np.asarray(labels)[valid]
            if len(coeff)<MIN_PHASE_TRIALS:continue
            z=np.mean(coeff); phase,ci_lo,ci_hi=bootstrap_phase_ci(coeff); mean_trace=np.nanmean(traces,axis=0); ph=window_slice(t,PHASE_WINDOW_S); tt=t[ph]; yy=mean_trace[ph]; peak_latency=float(tt[np.nanargmax(yy)]) if np.isfinite(yy).any() else np.nan; trough_latency=float(tt[np.nanargmin(yy)]) if np.isfinite(yy).any() else np.nan
            # Equal-weight image-identity coefficient, only relevant for ordinary image trials.
            balanced=np.nan+1j*np.nan
            if event_type=="image":
                pieces=[]
                for lab in pd.unique(labels_v):
                    cc=coeff[labels_v==lab]
                    if len(cc)>=MIN_IMAGE_TRIALS_PER_IDENTITY: pieces.append(np.mean(cc))
                if len(pieces)>=MIN_IDENTITIES_FOR_BALANCE: balanced=np.mean(np.asarray(pieces,complex))
            st=spikes[(spikes["session_id"].astype(str)==str(r.session_id))&(spikes["dmd"].astype(int)==int(r.dmd))&(spikes["roi"].astype(int)==int(r.roi))]["spike_time_sec"].to_numpy(float) if len(spikes) else np.array([],float)
            rel=[]
            for onset in onsets:
                q=st[(st>=onset)&(st<onset+STIMULUS_PERIOD_S)]-onset; rel.extend(q.tolist())
            rel=np.asarray(rel,float); angles=2*np.pi*rel/STIMULUS_PERIOD_S; spike_z=np.mean(np.exp(1j*angles)) if len(angles) else np.nan+1j*np.nan; spike_phase=float((np.angle(spike_z)/(2*np.pi)*STIMULUS_PERIOD_S)%STIMULUS_PERIOD_S) if len(angles) else np.nan
            phase_rows.append(dict(subject_id=str(r.subject_id),session_id=str(r.session_id),session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group),event_type=event_type,n_trials=len(coeff),
                                   dff_harmonic_amplitude=float(abs(z)),dff_phase_latency_s=phase,dff_phase_ci_delta_low_s=ci_lo,dff_phase_ci_delta_high_s=ci_hi,dff_phase_locking_p=phase_locking_p(coeff),dff_peak_latency_s=peak_latency,dff_trough_latency_s=trough_latency,
                                   identity_balanced_phase_latency_s=coeff_to_latency(balanced),identity_balanced_harmonic_amplitude=float(abs(balanced)) if np.isfinite(balanced.real) else np.nan,
                                   n_spikes_phase=len(rel),spike_vector_strength=float(abs(spike_z)) if len(rel) else np.nan,spike_phase_latency_s=spike_phase,spike_phase_p=rayleigh_p(angles)))

phase_df=pd.DataFrame(phase_rows)
for event_type in ["image","change","omission"]:
    idx=phase_df["event_type"].eq(event_type); phase_df.loc[idx,"dff_phase_locking_q"]=bh_fdr(phase_df.loc[idx,"dff_phase_locking_p"]); phase_df.loc[idx,"spike_phase_q"]=bh_fdr(phase_df.loc[idx,"spike_phase_p"])
phase_df["clear_dff_phasic"]=(phase_df["n_trials"]>=MIN_PHASE_TRIALS)&(phase_df["dff_phase_locking_q"]<FDR_ALPHA)
phase_df["clear_spike_phasic"]=(phase_df["n_spikes_phase"]>=MIN_SPIKES_FOR_PHASE)&(phase_df["spike_phase_q"]<FDR_ALPHA)

display(phase_df.head(12))
phase_df.to_csv(CACHE_DIR/"asap8_phase_cell_session_table.csv",index=False)


The first-harmonic phase is the preferred timing of the 1.33-Hz component relative to event onset. It should not be interpreted in isolation when harmonic amplitude is weak; `clear_dff_phasic` explicitly requires reproducible trial locking. `dff_peak_latency_s` is included because it is more intuitive, but it is expected to be noisier because it uses a single unsmoothed sample from the trial mean.


In [ ]:
# How many session × neuron observations are reproducibly phasic?
phase_counts=(phase_df.groupby(["session_label","event_type"],observed=True).agg(n_cells=("cell_id","size"),dff_phasic=("clear_dff_phasic","sum"),spike_phasic=("clear_spike_phasic","sum")).reset_index())
display(phase_counts)

fig,axs=plt.subplots(1,3,figsize=(10.5,3.1),sharey=True)
for ax,event_type in zip(axs,["image","change","omission"]):
    q=phase_counts[phase_counts["event_type"].eq(event_type)].set_index("session_label").reindex(SESSION_ORDER); x=np.arange(len(q)); ax.bar(x,q["dff_phasic"].fillna(0)); ax.set(xticks=x,xticklabels=SESSION_ORDER,title=event_type.capitalize(),xlabel="Session"); finish_axis(ax)
axs[0].set_ylabel("Clear phasic dF/F cells"); fig.tight_layout(); plt.show()


## Longitudinal phase shifts in manually registered cells


In [ ]:
def phase_shift_permutation(coeff_a,coeff_b,n_perm=N_PERM_SHIFT,seed=RNG_SEED):
    a=np.asarray(coeff_a,complex); b=np.asarray(coeff_b,complex); a=a[np.isfinite(a.real)&np.isfinite(a.imag)]; b=b[np.isfinite(b.real)&np.isfinite(b.imag)]
    if len(a)<MIN_PHASE_TRIALS or len(b)<MIN_PHASE_TRIALS:return (np.nan,np.nan)
    pa=coeff_to_latency(np.mean(a)); pb=coeff_to_latency(np.mean(b)); obs=float(circular_diff(pb,pa)); pool=np.concatenate([a,b]); n=len(a); rng=np.random.default_rng(seed); null=np.empty(int(n_perm))
    for i in range(int(n_perm)):
        perm=rng.permutation(len(pool)); za=np.mean(pool[perm[:n]]); zb=np.mean(pool[perm[n:]]); null[i]=circular_diff(coeff_to_latency(zb),coeff_to_latency(za))
    p=float((1+np.sum(np.abs(null)>=abs(obs)))/(len(null)+1)); return obs,p

shift_rows=[]
tracked=phase_df[phase_df["manually_registered"].astype(bool)&phase_df["global_cell_id"].astype(str).replace("nan","").ne("")].copy()
for (subject,cell,event_type),g in tracked.groupby(["subject_id","global_cell_id","event_type"],observed=True):
    g=g.sort_values("session_order")
    if g["session_id"].nunique()<2:continue
    # All observed pairs are retained; adjacent / first-last flags make focused plotting easy.
    rows=list(g.itertuples(index=False))
    for ia,ib in combinations(range(len(rows)),2):
        a,b=rows[ia],rows[ib]; sa=sessions[sessions["session_id"].astype(str).eq(str(a.session_id))].iloc[0]; sb=sessions[sessions["session_id"].astype(str).eq(str(b.session_id))].iloc[0]
        ta,tra,_,_=load_cell_trials(sa,a,event_type); tb,trb,_,_=load_cell_trials(sb,b,event_type); ca=trial_harmonic_coefficients(ta,tra); cb=trial_harmonic_coefficients(tb,trb); delta,p=phase_shift_permutation(ca,cb)
        shift_rows.append(dict(subject_id=str(subject),global_cell_id=str(cell),event_type=event_type,session_a=str(a.session_label),session_b=str(b.session_label),order_a=int(a.session_order),order_b=int(b.session_order),depth_um=float(np.nanmedian([a.depth_um,b.depth_um])),depth_group=str(a.depth_group),
                               phase_a_s=float(a.dff_phase_latency_s),phase_b_s=float(b.dff_phase_latency_s),phase_shift_s=delta,phase_shift_ms=1000*delta if np.isfinite(delta) else np.nan,phase_shift_p=p,both_dff_phasic=bool(a.clear_dff_phasic and b.clear_dff_phasic),
                               peak_latency_shift_ms=1000*(float(b.dff_peak_latency_s)-float(a.dff_peak_latency_s)),adjacent=(ib==ia+1),first_last=(ia==0 and ib==len(rows)-1)))

phase_shift_df=pd.DataFrame(shift_rows)
if len(phase_shift_df):
    for event_type in ["image","change","omission"]:
        idx=phase_shift_df["event_type"].eq(event_type)&phase_shift_df["both_dff_phasic"]
        phase_shift_df.loc[idx,"phase_shift_q"]=bh_fdr(phase_shift_df.loc[idx,"phase_shift_p"])
    phase_shift_df["clear_phase_shift"]=phase_shift_df["both_dff_phasic"]&(phase_shift_df["phase_shift_q"]<FDR_ALPHA)
else:
    phase_shift_df["clear_phase_shift"]=False

display(phase_shift_df.sort_values("phase_shift_q").head(20) if len(phase_shift_df) else phase_shift_df)
phase_shift_df.to_csv(CACHE_DIR/"asap8_longitudinal_phase_shifts.csv",index=False)


In [ ]:
# Same registered cells across sessions: preferred dF/F phase.
fig,axs=plt.subplots(1,3,figsize=(11,3.5),sharey=True)
for ax,event_type in zip(axs,["image","change","omission"]):
    q=tracked[(tracked["event_type"].eq(event_type))&tracked["clear_dff_phasic"]].copy(); xmap={s:i for i,s in enumerate(SESSION_ORDER)}
    for (subject,cell),g in q.groupby(["subject_id","global_cell_id"],observed=True):
        if g["session_id"].nunique()<2:continue
        g=g.assign(x=g["session_label"].map(xmap)).sort_values("x"); ax.plot(g["x"],1000*g["dff_phase_latency_s"],"-o",lw=.8,ms=3,alpha=.45,color=DEPTH_COLORS[str(g["depth_group"].iloc[0])])
    ax.set(xticks=np.arange(len(SESSION_ORDER)),xticklabels=SESSION_ORDER,xlabel="Session",title=event_type.capitalize(),ylim=(0,1000*STIMULUS_PERIOD_S)); finish_axis(ax)
axs[0].set_ylabel("Preferred dF/F phase (ms)"); fig.suptitle("Reproducibly phasic registered cells",y=1.02); fig.tight_layout(); plt.show()


In [ ]:
# Significant shifts only, with peak-latency change as a descriptive cross-check.
if len(phase_shift_df):
    sig=phase_shift_df[phase_shift_df["clear_phase_shift"]].copy()
    display(sig.sort_values(["event_type","phase_shift_q"]))
    fig,axs=plt.subplots(1,3,figsize=(10.5,3.2),sharex=True,sharey=True)
    for ax,event_type in zip(axs,["image","change","omission"]):
        q=phase_shift_df[(phase_shift_df["event_type"].eq(event_type))&phase_shift_df["both_dff_phasic"]&phase_shift_df["adjacent"]]
        ax.scatter(q["phase_shift_ms"],q["peak_latency_shift_ms"],s=25,alpha=.55); ax.axhline(0,color=".65",lw=.8); ax.axvline(0,color=".65",lw=.8); ax.set(title=event_type.capitalize(),xlabel="Harmonic phase shift (ms)"); finish_axis(ax)
    axs[0].set_ylabel("Unsmoothed peak-latency shift (ms)"); fig.tight_layout(); plt.show()


## Image-identity composition control


In [ ]:
q=phase_df[(phase_df["event_type"].eq("image"))&phase_df["identity_balanced_phase_latency_s"].notna()].copy()
fig,ax=plt.subplots(figsize=(4.2,4.0))
ax.scatter(1000*q["dff_phase_latency_s"],1000*q["identity_balanced_phase_latency_s"],s=28,alpha=.6)
ax.plot([0,750],[0,750],color=".6",ls="--",lw=1); ax.set(xlim=(0,750),ylim=(0,750),xlabel="All ordinary image trials phase (ms)",ylabel="Identity-balanced phase (ms)",title="Image phase vs equal-image weighting"); finish_axis(ax); fig.tight_layout(); plt.show()


## Trial-wise inspection of cells with the strongest longitudinal shifts


In [ ]:
def _phase_row(session_id,dmd,roi,event_type):
    q=phase_df[(phase_df["session_id"].astype(str)==str(session_id))&(phase_df["dmd"].astype(int)==int(dmd))&(phase_df["roi"].astype(int)==int(roi))&(phase_df["event_type"].astype(str)==event_type)]
    return q.iloc[0] if len(q) else None

def plot_phase_cell(subject_id,global_cell_id,event_type,max_trials=50):
    rr=rois[(rois["subject_id"].astype(str)==str(subject_id))&(rois["global_cell_id"].astype(str)==str(global_cell_id))&rois["included"].astype(bool)].sort_values("session_order")
    if rr.empty:raise ValueError(f"No registered cell {subject_id}:{global_cell_id}")
    rows=[]
    for r in rr.itertuples(index=False):
        p=_phase_row(r.session_id,r.dmd,r.roi,event_type)
        if p is not None: rows.append((r,p))
    if not rows:raise ValueError(f"No {event_type} phase data for {subject_id}:{global_cell_id}")
    fig,axs=plt.subplots(len(rows),3,figsize=(10.5,2.6*len(rows)),squeeze=False)
    for k,(r,p) in enumerate(rows):
        s=sessions[sessions["session_id"].astype(str).eq(str(r.session_id))].iloc[0]; t,traces,onsets,labels=load_cell_trials(s,r,event_type); take=np.arange(len(traces));
        if len(take)>max_trials:take=np.unique(np.linspace(0,len(traces)-1,max_trials).round().astype(int))
        mean=np.nanmean(traces,axis=0); c=DEPTH_COLORS[str(r.depth_group)]
        axs[k,0].plot(t,mean,color=c,lw=1.2); axs[k,0].axvline(0,color=".4",ls="--",lw=.8); axs[k,0].axvline(p.dff_phase_latency_s,color=EVENT_COLORS[event_type],lw=1.2,label="harmonic phase"); axs[k,0].axvline(p.dff_peak_latency_s,color="black",ls=":",lw=1,label="raw mean peak"); axs[k,0].set(xlim=(-.25,.80),ylabel="dF/F",title=f"{r.session_label} · phase={1000*p.dff_phase_latency_s:.0f} ms · q={p.dff_phase_locking_q:.2g}"); finish_axis(axs[k,0]);
        if k==0:axs[k,0].legend(frameon=False,fontsize=7)
        finite=traces[take][np.isfinite(traces[take])]; lim=np.nanpercentile(finite,[2,98]) if len(finite) else (-1,1); im=axs[k,1].imshow(traces[take],aspect="auto",interpolation="nearest",extent=[t[0],t[-1],len(take)-.5,-.5],vmin=lim[0],vmax=lim[1],cmap="viridis"); axs[k,1].axvline(0,color="white",ls="--",lw=.8); axs[k,1].axvline(p.dff_phase_latency_s,color="white",lw=.8); axs[k,1].set(xlim=(-.25,.80),ylabel="Trial",title="Raw trial dF/F")
        st=spikes[(spikes["session_id"].astype(str)==str(r.session_id))&(spikes["dmd"].astype(int)==int(r.dmd))&(spikes["roi"].astype(int)==int(r.roi))]["spike_time_sec"].to_numpy(float) if len(spikes) else np.array([])
        for j,onset in enumerate(np.asarray(onsets)[take]):
            rel=st[(st>=onset-.25)&(st<onset+.80)]-onset; axs[k,2].vlines(rel,j-.38,j+.38,color="black",lw=.65)
        axs[k,2].axvline(0,color=".4",ls="--",lw=.8); 
        if np.isfinite(p.spike_phase_latency_s):axs[k,2].axvline(p.spike_phase_latency_s,color=EVENT_COLORS[event_type],lw=1.1)
        axs[k,2].set(xlim=(-.25,.80),ylim=(len(take)-.5,-.5),ylabel="Trial",title=f"Spikes · preferred={1000*p.spike_phase_latency_s:.0f} ms" if np.isfinite(p.spike_phase_latency_s) else "Spikes"); finish_axis(axs[k,2])
    for ax in axs[-1,:]:ax.set_xlabel(f"Time from {event_type} (s)")
    fig.suptitle(f"M{subject_id} · registered cell {global_cell_id} · {event_type} phase across sessions",y=1.005); fig.tight_layout(); plt.show()


In [ ]:
N_PHASE_EXAMPLES = 2
for event_type in ["image","change","omission"]:
    q=phase_shift_df[(phase_shift_df["event_type"].eq(event_type))&phase_shift_df["clear_phase_shift"]].copy() if len(phase_shift_df) else pd.DataFrame()
    if q.empty:
        print(f"No FDR-significant {event_type} phase shifts under current criteria")
        continue
    q=q.assign(abs_shift=q["phase_shift_ms"].abs()).sort_values(["phase_shift_q","abs_shift"],ascending=[True,False]).drop_duplicates(["subject_id","global_cell_id"])
    for row in q.head(N_PHASE_EXAMPLES).itertuples(index=False):
        plot_phase_cell(row.subject_id,row.global_cell_id,event_type)


## Interpretation

A cell is most convincing as a phase-shifting example when (1) its event-locked harmonic component is reproducible within both sessions, (2) the session-label permutation rejects a common phase, (3) the raw trial heatmaps show the timing shift rather than a single-trial outlier, and (4) unsmoothed mean-peak latency and/or spike preferred timing move in the same general direction. For image responses, agreement between pooled and identity-balanced phase further argues that the shift is not simply caused by a different mixture of image identities across sessions.
